In [1]:
!sudo apt install swig

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.0.2-1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 20 not upgraded.


In [2]:
pip install finrl stable-baselines3 gym numpy pandas matplotlib

In [3]:
!pip install quantstats

In [5]:
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to /tmp/pip-req-build-asyt9wpi
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git /tmp/pip-req-build-asyt9wpi
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit 3a10805389c07e13e94333e94dcfe9beafce49b0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-5ghp2cm9/elegantrl_916f0dd9e4b548aebbdc923f3d884104
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-5ghp2cm9/elegantrl_916f0dd9e4b548aebbdc923f3d884104
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 62bf617fa87cf82b9fbf88307715d93b8ab8ee80
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.7/

In [6]:
import numpy as np
import pandas as pd
import gym
from stable_baselines3 import A2C
from stable_baselines3.common.vec_env import DummyVecEnv
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv

In [8]:
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.data_processor import DataProcessor

# Define tickers and time period
tickers = ["AAPL", "MSFT", "GOOG", "AMZN", "TSLA"]
start_date = "2020-01-01"
end_date = "2023-01-01"

# Download and process data
data = YahooDownloader(start_date=start_date, end_date=end_date, ticker_list=tickers).fetch_data()
# dp = DataProcessor(data)
# processed_data = dp.clean_data()

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (3780, 8)


In [9]:
data

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Price,date,close,high,low,open,volume,tic,day
0,2020-01-02,72.716057,75.150002,73.797501,74.059998,135480400,AAPL,3
1,2020-01-02,94.900497,94.900497,93.207497,93.750000,80580000,AMZN,3
2,2020-01-02,68.123718,68.406998,67.077499,67.077499,28132000,GOOG,3
3,2020-01-02,153.630722,160.729996,158.330002,158.779999,22622100,MSFT,3
4,2020-01-02,28.684000,28.713333,28.114000,28.299999,142981500,TSLA,3
...,...,...,...,...,...,...,...,...
3775,2022-12-30,128.436661,129.949997,127.430000,128.410004,77034200,AAPL,4
3776,2022-12-30,84.000000,84.050003,82.470001,83.120003,62401200,AMZN,4
3777,2022-12-30,88.412331,88.830002,87.029999,87.364998,19190300,GOOG,4
3778,2022-12-30,235.947845,239.960007,236.660004,238.210007,21938500,MSFT,4


In [13]:
!pip install shimmy

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [25]:
env_kwargs = {
    "df": data,
    "initial_amount": 100000,  # Initial capital
    "comission_fee_pct": 0.0025,  # Trading cost
    "reward_scaling": 1e-4,
    "time_window": 50,
    "normalize_df": None,
}

env = PortfolioOptimizationEnv(**env_kwargs)
# env = DummyVecEnv([lambda: env])  # Vectorized for SB3 compatibility


In [16]:
import warnings
warnings.filterwarnings('ignore')

In [26]:
model = A2C("MlpPolicy", env, verbose=1, learning_rate=0.0005, gamma=0.99)
model.learn(total_timesteps=1000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
-------------------------------------
| time/                 |           |
|    fps                | 79        |
|    iterations         | 100       |
|    time_elapsed       | 6         |
|    total_timesteps    | 500       |
| train/                |           |
|    entropy_loss       | -8.54     |
|    explained_variance | -1.19e-07 |
|    learning_rate      | 0.0005    |
|    n_updates          | 99        |
|    policy_loss        | 2.68e-05  |
|    std                | 1.01      |
|    value_loss         | 1.95e-11  |
-------------------------------------
Initial portfolio value:100000
Final portfolio value: 80611.28125
Final accumulative portfolio value: 0.8061128125
Maximum DrawDown: -0.605518531481593
Sharpe ratio: -0.11357216694116806


-------------------------------------
| rollout/              |           |
|    ep_len_mean        | 707       |
|    ep_rew_mean        | -2.2e-05  |
| time/                 |           |
|    fps                | 73        |
|    iterations         | 200       |
|    time_elapsed       | 13        |
|    total_timesteps    | 1000      |
| train/                |           |
|    entropy_loss       | -8.52     |
|    explained_variance | -1.19e-07 |
|    learning_rate      | 0.0005    |
|    n_updates          | 199       |
|    policy_loss        | -1.92e-05 |
|    std                | 1         |
|    value_loss         | 7.34e-12  |
-------------------------------------


In [27]:
import matplotlib.pyplot as plt

env._asset_memory["final"]
# portfolio_values = env.get_attr("history")[0]['portfolio_value']

# plt.plot(portfolio_values)
# plt.xlabel("Time Step")
# plt.ylabel("Portfolio Value")
# plt.title("Portfolio Performance Over Time")
# plt.show()

[100000,
 90001.61,
 94267.125,
 90017.984,
 92809.16,
 90063.09,
 89999.086,
 98140.15,
 97713.22,
 101268.62,
 98732.73,
 101462.41,
 101788.31,
 97798.75,
 98108.375,
 97609.58,
 103659.484,
 103817.38,
 105049.18,
 105860.555,
 109936.94,
 115892.28,
 116079.42,
 117976.484,
 117871.61,
 116481.52,
 112167.74,
 115530.29,
 115096.625,
 116830.27,
 120316.32,
 117677.82,
 121847.68,
 122569.76,
 118005.38,
 120270.86,
 120897.484,
 122082.65,
 122665.53,
 124228.75,
 124656.23,
 122771.38,
 121354.734,
 121969.02,
 122656.484,
 123802.336,
 123452.22,
 125121.94,
 124558.555,
 124328.05,
 123899.625,
 123879.75,
 123370.42,
 124706.4,
 125946.05,
 126174.29,
 126273.984,
 124875.55,
 126876.6,
 128447.04,
 129675.625,
 132529.8,
 127836.06,
 127498.9,
 129714.45,
 131454.92,
 131899.2,
 131977.7,
 131646.06,
 133011.17,
 134608.28,
 131519.8,
 132531.61,
 129555.31,
 131547.44,
 134243.61,
 136400.56,
 138669.6,
 144327.97,
 143370.89,
 145434.8,
 146673.72,
 149654.92,
 145995.66,
